# 04 Baseline Evaluation

Measure the zero-shot baseline of Gemma 4 2B-E2B on the ASL validation split before LoRA fine-tuning.

In [ ]:
from pathlib import Path

import torch
import yaml
from torch.utils.data import DataLoader

from src.data.pose_to_text_dataset import PoseToTextDataset, collate_pose_text_batch
from src.models.gemma_finetune import run_validation
from src.models.gemma_loader import load_gemma_4_2b_e2b

config = yaml.safe_load(Path("config.yaml").read_text(encoding="utf-8"))
device = "cuda" if torch.cuda.is_available() else "cpu"
config["data"]["training_pairs_dir"], device

In [ ]:
val_dataset = PoseToTextDataset.from_csv(
    Path(config["data"]["training_pairs_dir"]) / "val.csv",
    include_face=bool(config["data"]["include_face"]),
)
val_loader = DataLoader(
    val_dataset,
    batch_size=int(config["training"]["eval_batch_size"]),
    shuffle=False,
    num_workers=int(config["training"]["num_workers"]),
    collate_fn=collate_pose_text_batch,
)
len(val_dataset)

In [ ]:
model, tokenizer = load_gemma_4_2b_e2b(
    lora_rank=int(config["lora"]["rank"]),
    load_in_4bit=bool(config["gemma"]["load_in_4bit"]),
    max_seq_length=int(config["gemma"]["max_seq_length"]),
)
model = model.to(device) if hasattr(model, "to") else model

In [ ]:
metrics = run_validation(model, tokenizer, val_loader, device=torch.device(device))
print({
    "baseline_loss": metrics.loss,
    "baseline_accuracy": metrics.accuracy,
    "baseline_top5_accuracy": metrics.top5_accuracy,
    "target_expected_accuracy": 0.30,
})